# SAM copy-paste — prueba de humo reproducible

No usa IC-Light ni generación de cuadro completo. `static_vehicles.json` debe estar al mismo nivel que `train.csv` dentro del dataset `mtc-challenge`. La prueba está limitada a 10 llamadas SAM por clase, 2 inserciones por clase y 10 fondos; no modifica el dataset base.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

RUN_ID = 'sam_cp_smoke_v1'
ROOT = Path('/kaggle/working') / RUN_ID
if ROOT.exists(): shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total,memory.used,utilization.gpu', '--format=csv,noheader'], check=True)

REPO = ROOT / 'ia_article'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'chore/augmentation-evidence', '--filter=blob:none', '--sparse', 'https://github.com/unsa-semester-2026-A/ia_article.git', str(REPO)], check=True)
subprocess.run(['git', 'sparse-checkout', 'set', 'experiments'], cwd=REPO, check=True)
os.chdir(REPO / 'experiments')
# Kaggle already provides CUDA Torch and Ultralytics; do not reinstall either.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pytest', 'src/augmentation/test_masking.py', 'src/augmentation/test_copy_paste.py', 'src/augmentation/test_run.py', '-q'], check=True)


In [ ]:
input_root = Path('/kaggle/input')
dataset_root = next((p for p in (input_root / 'mtc-challenge', input_root / 'datasets/alvaroquispeunsa/mtc-challenge') if p.is_dir()), None)
if dataset_root is None: raise FileNotFoundError(f'MTC dataset not attached; input={[p.name for p in input_root.iterdir()]}')
static_json = dataset_root / 'static_vehicles.json'
if not static_json.is_file(): raise FileNotFoundError(f'Expected static_vehicles.json beside train.csv: {static_json}')
labels_root = dataset_root / 'yolo_obb_labels'
LABELS = labels_root / 'train' if (labels_root / 'train').is_dir() else labels_root
RAW = dataset_root / 'train_resized' / 'train'
LAMA = dataset_root / 'smart_lama_corrected' / 'train'
METADATA = dataset_root / 'split_metadata.csv'
for path in (LABELS, RAW, LAMA, METADATA, static_json):
    if not path.exists(): raise FileNotFoundError(path)
print({'dataset_root': str(dataset_root), 'static_json': str(static_json), 'labels': len(list(LABELS.glob('*.txt')))})


In [ ]:
WORK = ROOT / 'work'
DELTA = ROOT / 'delta'
EXPORTS = ROOT / 'exports'
prepare_cmd = [sys.executable, '-m', 'src.augmentation.run', 'prepare',
    '--split-metadata', str(METADATA), '--static-vehicles', str(static_json),
    '--labels-train', str(LABELS), '--raw-images', str(RAW), '--lama-images', str(LAMA),
    '--workdir', str(WORK), '--sam-model', 'sam_b.pt',
    '--max-source-tracks-per-class', '10', '--max-objects-per-class', '2', '--max-jobs', '10',
    '--no-drive-sync']
subprocess.run(prepare_cmd, check=True)
subprocess.run([sys.executable, '-m', 'src.augmentation.run', 'render', '--jobs-jsonl', str(WORK/'jobs.jsonl'), '--output-dir', str(DELTA), '--no-drive-sync'], check=True)
subprocess.run([sys.executable, '-m', 'src.augmentation.run', 'package-delta', '--synthetic-images', str(DELTA/'images'), '--synthetic-labels', str(DELTA/'labels'), '--manifest', str(DELTA/'manifest.csv'), '--output-dir', str(EXPORTS), '--run-id', RUN_ID, '--no-drive-sync'], check=True)
assert list((DELTA/'images').glob('*.jpg')), 'No synthetic image was rendered'
assert (EXPORTS/f'sam_copy_paste_delta_{RUN_ID}.zip').is_file()
print('PASSED:', RUN_ID, 'images=', len(list((DELTA/'images').glob('*.jpg'))), 'output=', ROOT)
